In [ ]:
!pip install transformers datasets evaluate accelerate -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset

In [ ]:
df = pd.read_csv("/content/sarcasm_mixed_reviews_1000.csv")

df = df[["reviews", "style"]].dropna()
df = df.rename(columns={"reviews": "text", "style": "label"})

print(df.head())
print(df["label"].value_counts())

In [ ]:
label2id = {"standard": 0, "sarcastic": 1}
id2label = {0: "standard", 1: "sarcastic"}

df["label"] = df["label"].map(label2id)

In [ ]:
df.head()

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
test_dataset

In [ ]:
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_sarcasm",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch", # Added to match eval_strategy
    learning_rate=2e-5,
    load_best_model_at_end=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results=trainer.evaluate()
print(results)

In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=["standard", "sarcastic"]))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_true, y_pred))

In [ ]:
import torch

def predict_sarcasm(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return id2label[pred]

# YOUR TEST CASES
sample_reviews = [
    "This product is fantastic and works perfectly.",
    "Wow, amazing charger. It stopped working in one day.",
    "Absolutely brilliant. I love how it stopped working immediately.",
    "The product is average, not too bad and not too good.",
    "disappointing charger. I thought it would stop working in one day, but is working absolutely fine."
]

for review in sample_reviews:
    print("Review:", review)
    print("Predicted:", predict_sarcasm(review))
    print("-"*60)


Testing zero shot bert for sarcasm

In [ ]:
!pip install transformers -q

from transformers import pipeline
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

In [ ]:
df = pd.read_csv("/content/sarcasm_mixed_reviews_1000.csv")

df = df[["reviews", "style"]].dropna()

print(df.head())

In [ ]:
labels = ["sarcastic", "standard"]

In [ ]:
sample_reviews = [
    "This product is fantastic and works perfectly.",
    "Wow, amazing charger. It stopped working in one day.",
    "Absolutely brilliant. I love how it stopped working immediately.",
    "The product is average, not too bad and not too good.",
    "disappointing charger. I thought it would stop working in one day, but is working absolutely fine."
]

for r in sample_reviews:
    result = classifier(r, labels)
    print("Review:", r)
    print("Prediction:", result["labels"][0])
    print("Scores:", dict(zip(result["labels"], result["scores"])))
    print("-"*60)

In [ ]:
predictions = []

for text in df["reviews"]:
    result = classifier(text, labels)
    predictions.append(result["labels"][0])

In [ ]:
print("Accuracy:", accuracy_score(df["style"], predictions))

print("\nClassification Report:\n")
print(classification_report(df["style"], predictions))